
# Buy Fee Large Dataset Audit & Modeling Package

## Problem statement

The full VIN-level Buy Fee dataset is approximately **1.4 GB**, so loading the entire CSV into memory at once is inefficient and may fail on a normal corporate laptop.

This notebook reads the file **in chunks** and creates a compact review package for Buy Fee attrition / elasticity modeling.

### Outputs

1. `DATA_PROFILE.csv` — column-level profile.
2. `FEE_CHANGE_EVENTS.csv` — deduplicated historical fee-change events using `BUYFEE_CHANGE_STATUS == "Change"`.
3. `WEEKLY_MODEL_PANEL.csv` — weekly Location × Lane × Price Tier panel.
4. `RANDOM_SAMPLE.csv` — random VIN-level sample.
5. `CHANGE_EVENT_SAMPLE.csv` — VIN-level records within ±12 weeks of real fee-change events.
6. `COLUMN_SUMMARY.csv` — dataset-level audit summary.

### Business rules used

- A fee-change event is created only when `BUYFEE_CHANGE_STATUS == "Change"`.
- Event grain: **Location × Lane × Price Tier × Change Date**.
- AAA sold logic:
  - `VRSTATUS == "HST"` and `VRINVOICE != "RETURN"`, OR
  - `VRSTATUS == "SLD"` and `VRIFORTRY != "Y"`.
- Price tiers default to **$1,000 bands**.
- CSV encoding is detected automatically.


In [3]:

from pathlib import Path
import pandas as pd
import numpy as np
import math
import gc
import warnings
from collections import Counter

warnings.filterwarnings("ignore")

# ============================================================
# USER CONFIGURATION
# ============================================================
DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

# Change ONLY this filename if needed.
INPUT_CSV = DATA_FOLDER / "VIN_LEVEL_DATA_WITH_DATES_AND_SALE_PRICE_NEW_DATA.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_review_package"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 250_000
RANDOM_SAMPLE_SIZE = 100_000
CHANGE_WINDOW_WEEKS = 12
PRICE_TIER_WIDTH = 1000
RANDOM_STATE = 42

# Optional hard overrides. Leave None for auto-detection.
COLUMN_OVERRIDES = {
    "vin": None,
    "status": None,
    "sale_date": None,
    "sale_amount": None,
    "location": None,
    "lane": None,
    "invoice": None,
    "ifortry": None,
    "buyfee_final": None,
    "buyer_buyfee": None,
    "change_status": None,
    "change_date": None,
    "previous_buyfee": None,
    "change_amount": None,
}

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print("Input file :", INPUT_CSV)
print("Exists     :", INPUT_CSV.exists())
print("Output dir :", OUTPUT_DIR)
print("Chunk size :", f"{CHUNK_SIZE:,}")

if not INPUT_CSV.exists():
    print("\nCSV files found in the configured folder:")
    for f in DATA_FOLDER.glob("*.csv"):
        print(" -", f.name)
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")


CONFIGURATION
Input file : C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\VIN_LEVEL_DATA_WITH_DATES_AND_SALE_PRICE_NEW_DATA.csv
Exists     : True
Output dir : C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_review_package
Chunk size : 250,000


## 1. Detect encoding and map columns

In [4]:

ENCODINGS = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

def detect_encoding(path):
    last_err = None
    for enc in ENCODINGS:
        try:
            pd.read_csv(path, nrows=5, encoding=enc)
            return enc
        except UnicodeDecodeError as exc:
            last_err = exc
    raise last_err

ENCODING = detect_encoding(INPUT_CSV)
header_df = pd.read_csv(INPUT_CSV, nrows=5, encoding=ENCODING, low_memory=False)

print("Encoding:", ENCODING)
print("Columns found:", len(header_df.columns))
display(pd.DataFrame({"Column": header_df.columns}))

ALIASES = {
    "vin": ["VRVIN", "VIN", "vin"],
    "status": ["VRSTATUS", "STATUS", "status"],
    "sale_date": ["VRSALEDATE", "SALEDATE", "SaleDate", "sale_date", "SALE_DATE"],
    "sale_amount": ["VRSALEAMT", "SALEAMT_FINAL", "Sale_Amount", "SALE_AMOUNT", "sale_amount"],
    "location": ["VRLOCATION", "Location", "LOCATION", "Auction", "AUCTION"],
    "lane": ["LANE", "Lane", "ACTUAL_LANE", "Actual Lane", "Actual_Lane"],
    "invoice": ["VRINVOICE", "INVOICE", "Invoice"],
    "ifortry": ["VRIFORTRY", "IFORTRY", "IForTry"],
    "buyfee_final": ["BUYFEE_FINAL", "BUYFEE-FINAL", "BUYFEE FINAL", "Buyer_BuyFee_Final"],
    "buyer_buyfee": ["Buyer_BuyFee", "BUYER_BUYFEE", "BuyerBuyFee"],
    "change_status": ["BUYFEE_CHANGE_STATUS", "BUY_FEE_CHANGE_STATUS", "BuyFee_Change_Status"],
    "change_date": ["BUYFEE_CHANGE_DATE", "BUY_FEE_CHANGE_DATE", "BuyFee_Change_Date"],
    "previous_buyfee": ["PREVIOUS_BUYFEE", "PREVIOUS_BUY_FEE", "Previous_BuyFee"],
    "change_amount": ["BUYFEE_CHANGE_AMOUNT", "BUY_FEE_CHANGE_AMOUNT", "BuyFee_Change_Amount"],
}

def resolve_col(columns, logical_name):
    override = COLUMN_OVERRIDES.get(logical_name)
    if override:
        if override not in columns:
            raise ValueError(f"Override for {logical_name} not found: {override}")
        return override
    lookup = {str(c).strip().upper(): c for c in columns}
    for alias in ALIASES[logical_name]:
        key = alias.strip().upper()
        if key in lookup:
            return lookup[key]
    return None

COL = {k: resolve_col(header_df.columns, k) for k in ALIASES}

display(pd.DataFrame([{"Logical_Field": k, "Resolved_Column": v} for k, v in COL.items()]))

required = ["sale_date", "sale_amount", "location", "lane", "change_status", "change_date", "previous_buyfee", "change_amount"]
missing_required = [k for k in required if COL[k] is None]
if missing_required:
    raise ValueError(
        "Missing required logical fields: " + ", ".join(missing_required) +
        "\nUse COLUMN_OVERRIDES in the first cell to map them manually."
    )


Encoding: utf-8
Columns found: 36


,Column
0,VRVIN
1,VRSTATUS
2,VRSALEAMT
3,VRMILEAGE
4,VRLOCATION
5,VRINVOICE
6,VRIFORTRY
7,VRHIGHBID
8,VRFLOOR
9,VRBUYER


,Logical_Field,Resolved_Column
0,vin,VRVIN
1,status,VRSTATUS
2,sale_date,VRSALEDATE
3,sale_amount,VRSALEAMT
4,location,VRLOCATION
5,lane,LANE
6,invoice,VRINVOICE
7,ifortry,VRIFORTRY
8,buyfee_final,BUYFEE-FINAL
9,buyer_buyfee,Buyer_BuyFee


## 2. Helper functions

In [5]:

def parse_date_series(s):
    s2 = s.astype("string").str.strip()
    compact = s2.str.replace(r"\.0$", "", regex=True)
    mask8 = compact.str.fullmatch(r"\d{8}", na=False)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    if mask8.any():
        out.loc[mask8] = pd.to_datetime(compact.loc[mask8], format="%Y%m%d", errors="coerce")
    rem = ~mask8
    if rem.any():
        out.loc[rem] = pd.to_datetime(s2.loc[rem], errors="coerce")
    return out

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def clean_text(s):
    return s.astype("string").str.strip()

def normalize_change_status(s):
    return clean_text(s).str.upper()

def make_price_tier(sale_amt):
    amt = pd.to_numeric(sale_amt, errors="coerce")
    low = np.floor(amt / PRICE_TIER_WIDTH) * PRICE_TIER_WIDTH
    high = low + PRICE_TIER_WIDTH
    low_i = pd.Series(low, index=sale_amt.index).astype("Int64")
    high_i = pd.Series(high, index=sale_amt.index).astype("Int64")
    label = pd.Series(pd.NA, index=sale_amt.index, dtype="string")
    valid = amt.notna() & (amt >= 0)
    label.loc[valid] = low_i.loc[valid].astype(str) + "-" + high_i.loc[valid].astype(str)
    return low, high, label

def sold_flag(df):
    status = clean_text(df[COL["status"]]).str.upper() if COL["status"] else pd.Series("", index=df.index)
    invoice = clean_text(df[COL["invoice"]]).str.upper() if COL["invoice"] else pd.Series("", index=df.index)
    ifortry = clean_text(df[COL["ifortry"]]).str.upper() if COL["ifortry"] else pd.Series("", index=df.index)

    hst = status.eq("HST")
    if COL["invoice"]:
        hst &= ~invoice.eq("RETURN")

    sld = status.eq("SLD")
    if COL["ifortry"]:
        sld &= ~ifortry.eq("Y")

    return hst | sld


## 3. Pass 1 — profile, fee events, weekly panel, random sample

In [8]:
# ============================================================
# ROBUST LARGE CSV CHUNK READER
# ============================================================

def get_working_encoding(path, test_rows=100_000):

    encodings_to_try = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    print("=" * 70)
    print("ENCODING TEST")
    print("=" * 70)

    for enc in encodings_to_try:

        try:

            test = pd.read_csv(
                path,
                nrows=test_rows,
                encoding=enc,
                low_memory=False
            )

            print(
                f"Successfully tested encoding: {enc}"
            )

            del test
            gc.collect()

            return enc

        except UnicodeDecodeError:

            print(
                f"Failed encoding: {enc}"
            )

        except Exception as exc:

            print(
                f"Encoding {enc} produced another error:"
            )

            print(exc)

    # latin1 can decode every byte value,
    # so this is a final defensive fallback.
    print(
        "\nUsing latin1 as final fallback."
    )

    return "latin1"


# ============================================================
# DETERMINE SAFE ENCODING
# ============================================================

SAFE_ENCODING = get_working_encoding(
    INPUT_CSV
)

print(
    "\nSelected encoding:",
    SAFE_ENCODING
)


# ============================================================
# CREATE ROBUST CHUNK READER
# ============================================================

reader = pd.read_csv(
    INPUT_CSV,
    chunksize=CHUNK_SIZE,
    encoding=SAFE_ENCODING,
    encoding_errors="replace",
    low_memory=False
)


# ============================================================
# PROCESS CHUNKS
# ============================================================

for chunk in reader:

    chunk_count += 1

    n = len(chunk)

    total_rows += n


    # ========================================================
    # 1. COLUMN PROFILE
    # ========================================================

    for c in chunk.columns:

        s = chunk[c]
        p = profile[c]

        nn = s.notna().sum()

        p["non_null"] += int(nn)
        p["null"] += int(n - nn)


        # ----------------------------------------------------
        # Sample values
        # ----------------------------------------------------

        if len(p["sample_values"]) < SAMPLE_VALUES_PER_COL:

            vals = (
                s.dropna()
                .astype(str)
                .drop_duplicates()
                .head(
                    SAMPLE_VALUES_PER_COL
                    - len(p["sample_values"])
                )
                .tolist()
            )

            p["sample_values"].extend(vals)


        # ----------------------------------------------------
        # Approx unique values
        # ----------------------------------------------------

        if not p["unique_capped"]:

            vals = (
                s.dropna()
                .astype(str)
                .drop_duplicates()
            )

            remaining = (
                UNIQUE_TRACK_CAP
                - len(p["unique_values"])
            )

            if remaining > 0:

                p["unique_values"].update(
                    vals.head(remaining).tolist()
                )

            if len(p["unique_values"]) >= UNIQUE_TRACK_CAP:

                p["unique_capped"] = True


        # ----------------------------------------------------
        # Numeric profile
        # ----------------------------------------------------

        num = pd.to_numeric(
            s,
            errors="coerce"
        )

        valid = num.dropna()

        if len(valid):

            p["numeric_count"] += len(valid)

            p["numeric_sum"] += float(
                valid.sum()
            )

            p["numeric_sumsq"] += float(
                np.square(
                    valid.astype(float)
                ).sum()
            )

            p["numeric_min"] = min(
                p["numeric_min"],
                float(valid.min())
            )

            p["numeric_max"] = max(
                p["numeric_max"],
                float(valid.max())
            )


    # ========================================================
    # 2. STANDARDIZE IMPORTANT FIELDS
    # ========================================================

    sale_dt = parse_date_series(
        chunk[COL["sale_date"]]
    )

    sale_amt = to_num(
        chunk[COL["sale_amount"]]
    )

    loc = (
        clean_text(
            chunk[COL["location"]]
        )
        .str.upper()
    )

    lane = (
        clean_text(
            chunk[COL["lane"]]
        )
        .str.upper()
    )

    change_status = normalize_change_status(
        chunk[COL["change_status"]]
    )

    change_dt = parse_date_series(
        chunk[COL["change_date"]]
    )

    prev_fee = to_num(
        chunk[COL["previous_buyfee"]]
    )

    change_amt = to_num(
        chunk[COL["change_amount"]]
    )


    # ========================================================
    # 3. PRICE TIERS
    # ========================================================

    tier_low, tier_high, tier_label = (
        make_price_tier(
            sale_amt
        )
    )


    # ========================================================
    # 4. SOLD FLAG
    # ========================================================

    # IMPORTANT FIX:
    # sold_flag() can return nullable boolean values.
    # Missing values must become False before conversion.

    is_sold = sold_flag(chunk)

    is_sold = (
        pd.Series(
            is_sold,
            index=chunk.index
        )
        .fillna(False)
        .astype(bool)
    )


    # ========================================================
    # 5. BUY FEE
    # ========================================================

    if COL["buyfee_final"]:

        buyfee = to_num(
            chunk[COL["buyfee_final"]]
        )

    elif COL["buyer_buyfee"]:

        buyfee = to_num(
            chunk[COL["buyer_buyfee"]]
        )

    else:

        buyfee = (
            prev_fee
            + change_amt
        )


    # ========================================================
    # 6. GLOBAL DATASET SUMMARY
    # ========================================================

    global_location_values.update(
        loc.dropna()
        .unique()
        .tolist()
    )

    global_lane_values.update(
        lane.dropna()
        .unique()
        .tolist()
    )

    change_status_counter.update(
        change_status
        .dropna()
        .tolist()
    )


    # --------------------------------------------------------
    # VIN count
    # --------------------------------------------------------

    if COL["vin"] and not vin_unique_capped:

        vals = (
            clean_text(
                chunk[COL["vin"]]
            )
            .dropna()
            .unique()
            .tolist()
        )

        remaining = (
            VIN_UNIQUE_CAP
            - len(global_vin_values)
        )

        global_vin_values.update(
            vals[:max(0, remaining)]
        )

        if (
            len(global_vin_values)
            >= VIN_UNIQUE_CAP
        ):

            vin_unique_capped = True


    # --------------------------------------------------------
    # Sale date range
    # --------------------------------------------------------

    if sale_dt.notna().any():

        mn = sale_dt.min()
        mx = sale_dt.max()

        min_sale_date = (
            mn
            if min_sale_date is None
            else min(min_sale_date, mn)
        )

        max_sale_date = (
            mx
            if max_sale_date is None
            else max(max_sale_date, mx)
        )


    # --------------------------------------------------------
    # Buy fee change date range
    # --------------------------------------------------------

    if change_dt.notna().any():

        mn = change_dt.min()
        mx = change_dt.max()

        min_change_date = (
            mn
            if min_change_date is None
            else min(min_change_date, mn)
        )

        max_change_date = (
            mx
            if max_change_date is None
            else max(max_change_date, mx)
        )


    # ========================================================
    # 7. RANDOM SAMPLE
    # ========================================================

    take_n = min(
        n,
        5000
    )

    if take_n > 0:

        sample_parts.append(
            chunk.sample(
                n=take_n,
                random_state=(
                    RANDOM_STATE
                    + chunk_count
                )
            )
        )


    # ========================================================
    # 8. TRUE BUY FEE CHANGE EVENTS
    # ========================================================

    # IMPORTANT:
    # Only BUYFEE_CHANGE_STATUS == CHANGE
    # is considered an actual fee-change event.

    is_change = (
        change_status
        .eq("CHANGE")
        .fillna(False)
    )


    if is_change.any():

        ev = pd.DataFrame({
            "VRLOCATION":
                loc.loc[is_change],

            "LANE":
                lane.loc[is_change],

            "price_tier":
                tier_label.loc[is_change],

            "price_tier_low":
                tier_low.loc[is_change],

            "price_tier_high":
                tier_high.loc[is_change],

            "BUYFEE_CHANGE_DATE":
                change_dt.loc[is_change],

            "PREVIOUS_BUYFEE":
                prev_fee.loc[is_change],

            "BUYFEE_CHANGE_AMOUNT":
                change_amt.loc[is_change],
        })


        ev["NEW_BUYFEE"] = (
            ev["PREVIOUS_BUYFEE"]
            + ev["BUYFEE_CHANGE_AMOUNT"]
        )


        ev = ev.dropna(
            subset=[
                "VRLOCATION",
                "LANE",
                "price_tier",
                "BUYFEE_CHANGE_DATE"
            ]
        )


        event_parts.append(
            ev
        )


    # ========================================================
    # 9. CREATE WEEK START
    # ========================================================

    week_start = (
        sale_dt
        - pd.to_timedelta(
            sale_dt.dt.weekday,
            unit="D"
        )
    )


    # ========================================================
    # 10. SAFE NUMERIC ARRAYS
    # ========================================================

    sold_int = (
        is_sold
        .astype("int8")
    )


    sale_amt_sold = (
        sale_amt
        .where(
            is_sold,
            np.nan
        )
    )


    buyfee_sold = (
        buyfee
        .where(
            is_sold,
            np.nan
        )
    )


    # ========================================================
    # 11. WEEKLY TEMP DATA
    # ========================================================

    temp = pd.DataFrame({

        "Week_Start":
            week_start,

        "VRLOCATION":
            loc,

        "LANE":
            lane,

        "price_tier":
            tier_label,

        "price_tier_low":
            tier_low,

        "price_tier_high":
            tier_high,

        "Registered_Units":
            np.ones(
                len(chunk),
                dtype=np.int8
            ),

        "Sold_Units":
            sold_int,

        "Sale_Amount_Sold":
            sale_amt_sold,

        "BuyFee_Sold":
            buyfee_sold,
    })


    temp = temp.dropna(
        subset=[
            "Week_Start",
            "VRLOCATION",
            "LANE",
            "price_tier"
        ]
    )


    # ========================================================
    # 12. WEEKLY AGGREGATION
    # ========================================================

    weekly = (

        temp.groupby(

            [
                "Week_Start",
                "VRLOCATION",
                "LANE",
                "price_tier",
                "price_tier_low",
                "price_tier_high"
            ],

            dropna=False,
            sort=False
        )

        .agg(

            Registered_Units=(
                "Registered_Units",
                "sum"
            ),

            Sold_Units=(
                "Sold_Units",
                "sum"
            ),

            Sale_Amount_Sum=(
                "Sale_Amount_Sold",
                "sum"
            ),

            Sale_Amount_Count=(
                "Sale_Amount_Sold",
                "count"
            ),

            BuyFee_Revenue=(
                "BuyFee_Sold",
                "sum"
            ),

            BuyFee_Count=(
                "BuyFee_Sold",
                "count"
            ),
        )

        .reset_index()
    )


    weekly_parts.append(
        weekly
    )


    # ========================================================
    # 13. PROGRESS REPORT
    # ========================================================

    if chunk_count % 5 == 0:

        print(
            f"Processed {chunk_count:,} chunks "
            f"| {total_rows:,} rows"
        )


    # ========================================================
    # 14. FREE MEMORY
    # ========================================================

    del (
        chunk,
        temp,
        weekly,
        sold_int,
        sale_amt_sold,
        buyfee_sold
    )

    gc.collect()


# ============================================================
# PASS 1 COMPLETE
# ============================================================

print("\n" + "=" * 70)

print(
    "PASS 1 COMPLETE"
)

print("=" * 70)

print(
    "Chunks:",
    f"{chunk_count:,}"
)

print(
    "Rows:",
    f"{total_rows:,}"
)

print(
    "Fee-change source chunks:",
    f"{len(event_parts):,}"
)

print(
    "Weekly aggregation chunks:",
    f"{len(weekly_parts):,}"
)

ENCODING TEST
Successfully tested encoding: utf-8

Selected encoding: utf-8
Processed 10 chunks | 2,500,000 rows
Processed 15 chunks | 3,750,000 rows
Processed 20 chunks | 5,000,000 rows
Processed 25 chunks | 6,250,000 rows
Processed 30 chunks | 7,500,000 rows

PASS 1 COMPLETE
Chunks: 31
Rows: 7,540,598
Fee-change source chunks: 31
Weekly aggregation chunks: 31


## 4. Save `DATA_PROFILE.csv`

In [9]:

profile_rows = []
for c, p in profile.items():
    non_null = p["non_null"]
    null = p["null"]
    total = non_null + null
    num_count = p["numeric_count"]
    num_mean = p["numeric_sum"] / num_count if num_count else np.nan
    if num_count > 1:
        variance = (p["numeric_sumsq"] / num_count) - (num_mean ** 2)
        num_std = math.sqrt(max(0, variance))
    else:
        num_std = np.nan

    profile_rows.append({
        "Column": c,
        "Observed_Dtype_Sample": str(header_df[c].dtype),
        "Rows": total,
        "NonNull_Count": non_null,
        "Missing_Count": null,
        "Missing_Pct": (null / total) if total else np.nan,
        "Approx_Unique_Count": f">={UNIQUE_TRACK_CAP:,}" if p["unique_capped"] else len(p["unique_values"]),
        "Numeric_Convertible_Count": num_count,
        "Numeric_Convertible_Pct": (num_count / non_null) if non_null else np.nan,
        "Numeric_Min": np.nan if p["numeric_min"] == np.inf else p["numeric_min"],
        "Numeric_Max": np.nan if p["numeric_max"] == -np.inf else p["numeric_max"],
        "Numeric_Mean": num_mean,
        "Numeric_Std": num_std,
        "Sample_Values": " | ".join(map(str, p["sample_values"][:SAMPLE_VALUES_PER_COL]))
    })

data_profile = pd.DataFrame(profile_rows)
profile_path = OUTPUT_DIR / "DATA_PROFILE.csv"
data_profile.to_csv(profile_path, index=False)
print("Saved:", profile_path)
display(data_profile.head(20))


Saved: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_review_package\DATA_PROFILE.csv


,Column,Observed_Dtype_Sample,Rows,NonNull_Count,Missing_Count,Missing_Pct,Approx_Unique_Count,Numeric_Convertible_Count,Numeric_Convertible_Pct,Numeric_Min,Numeric_Max,Numeric_Mean,Numeric_Std,Sample_Values
0,VRVIN,str,7540598,7540598,0,0.000000,">=100,000",112,0.000015,1.000000e+16,9.440400e+16,2.188658e+16,2.018419e+16,10000000000001492 | 10000000000003665 | 100000...
1,VRSTATUS,str,7540598,7540598,0,0.000000,5,0,0.000000,NaN,NaN,NaN,NaN,HST | NOS | INV | SAL | SLD
2,VRSALEAMT,int64,7540598,7531341,9257,0.001228,7073,7531341,1.000000,0.000000e+00,9.999990e+05,6.627843e+03,1.091684e+04,750.0 | 1000.0 | 200.0 | 11500.0 | 0.0
3,VRMILEAGE,int64,7540598,7537326,3272,0.000434,">=100,000",7537326,1.000000,0.000000e+00,9.999990e+05,1.193964e+05,7.198496e+04,404255.0 | 232846.0 | 1.0 | 0.0 | 4573.0
4,VRLOCATION,str,7540598,7540598,0,0.000000,54,0,0.000000,NaN,NaN,NaN,NaN,BADGERSTATE | SOUTHWISCONSIN | HARRISBURG | SA...
5,VRINVOICE,float64,7540598,413041,7127557,0.945224,1255,1786,0.004324,1.636400e+04,2.265700e+05,7.124254e+04,8.302427e+04,| RETURN | 028553 | 028250 | 016416
6,VRIFORTRY,str,7540598,4549905,2990693,0.396612,5,0,0.000000,NaN,NaN,NaN,NaN,N | | H | N |
7,VRHIGHBID,int64,7540598,7540598,0,0.000000,7158,7540598,1.000000,0.000000e+00,9.999990e+05,8.460178e+03,1.166321e+04,750 | 1000 | 200 | 11500 | 0
8,VRFLOOR,int64,7540598,7540598,0,0.000000,18489,7540598,1.000000,-3.042000e+04,9.999990e+05,7.857504e+03,1.705359e+04,0 | 200 | 15000 | 700 | 26500
9,VRBUYER,str,7540598,4540567,3000031,0.397851,69567,369699,0.081421,1.000000e+00,9.034660e+05,1.929276e+04,1.015487e+05,J255 | P194 | B0005 | A01427 | C0654


## 5. Save `FEE_CHANGE_EVENTS.csv`

In [10]:

if event_parts:
    fee_events = pd.concat(event_parts, ignore_index=True)
else:
    fee_events = pd.DataFrame(columns=[
        "VRLOCATION", "LANE", "price_tier", "price_tier_low", "price_tier_high",
        "BUYFEE_CHANGE_DATE", "PREVIOUS_BUYFEE", "BUYFEE_CHANGE_AMOUNT", "NEW_BUYFEE"
    ])

fee_events["BUYFEE_CHANGE_DATE"] = pd.to_datetime(fee_events["BUYFEE_CHANGE_DATE"], errors="coerce")
event_key = ["VRLOCATION", "LANE", "price_tier", "BUYFEE_CHANGE_DATE"]

fee_events = (
    fee_events.groupby(event_key, dropna=False, as_index=False)
    .agg(
        price_tier_low=("price_tier_low", "first"),
        price_tier_high=("price_tier_high", "first"),
        PREVIOUS_BUYFEE=("PREVIOUS_BUYFEE", "median"),
        BUYFEE_CHANGE_AMOUNT=("BUYFEE_CHANGE_AMOUNT", "median"),
        NEW_BUYFEE=("NEW_BUYFEE", "median"),
        Source_Rows=("NEW_BUYFEE", "size"),
    )
)

fee_events["FEE_CHANGE_PCT"] = np.where(
    fee_events["PREVIOUS_BUYFEE"].abs() > 0,
    fee_events["BUYFEE_CHANGE_AMOUNT"] / fee_events["PREVIOUS_BUYFEE"],
    np.nan
)
fee_events["FEE_DIRECTION"] = np.select(
    [fee_events["BUYFEE_CHANGE_AMOUNT"] > 0, fee_events["BUYFEE_CHANGE_AMOUNT"] < 0],
    ["Increase", "Decrease"],
    default="No Change"
)

events_path = OUTPUT_DIR / "FEE_CHANGE_EVENTS.csv"
fee_events.to_csv(events_path, index=False)

print("Saved:", events_path)
print("Unique fee-change events:", f"{len(fee_events):,}")
print("Unique location × change dates:", f"{fee_events[['VRLOCATION','BUYFEE_CHANGE_DATE']].drop_duplicates().shape[0]:,}")
print("Unique company change dates:", fee_events["BUYFEE_CHANGE_DATE"].nunique())
display(fee_events.head())


Saved: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_review_package\FEE_CHANGE_EVENTS.csv
Unique fee-change events: 7,754
Unique location × change dates: 103
Unique company change dates: 51


,VRLOCATION,LANE,price_tier,BUYFEE_CHANGE_DATE,price_tier_low,price_tier_high,PREVIOUS_BUYFEE,BUYFEE_CHANGE_AMOUNT,NEW_BUYFEE,Source_Rows,FEE_CHANGE_PCT,FEE_DIRECTION
0,ATLANTA,A,0-1000,2024-12-01,0.0,1000.0,360.0,40.0,400.0,68,0.111111,Increase
1,ATLANTA,A,0-1000,2025-06-08,0.0,1000.0,600.0,125.0,725.0,4,0.208333,Increase
2,ATLANTA,A,1000-2000,2024-12-01,1000.0,2000.0,250.0,30.0,280.0,10,0.120000,Increase
3,ATLANTA,A,10000-11000,2024-12-01,10000.0,11000.0,460.0,50.0,510.0,6,0.108696,Increase
4,ATLANTA,A,11000-12000,2024-12-01,11000.0,12000.0,475.0,50.0,525.0,4,0.105263,Increase


## 6. Save `WEEKLY_MODEL_PANEL.csv`

In [11]:

weekly_all = pd.concat(weekly_parts, ignore_index=True)
group_cols = ["Week_Start", "VRLOCATION", "LANE", "price_tier", "price_tier_low", "price_tier_high"]

weekly_panel = (
    weekly_all.groupby(group_cols, dropna=False, sort=False)
    .agg(
        Registered_Units=("Registered_Units", "sum"),
        Sold_Units=("Sold_Units", "sum"),
        Sale_Amount_Sum=("Sale_Amount_Sum", "sum"),
        Sale_Amount_Count=("Sale_Amount_Count", "sum"),
        BuyFee_Revenue=("BuyFee_Revenue", "sum"),
        BuyFee_Count=("BuyFee_Count", "sum"),
    )
    .reset_index()
)

weekly_panel["Conversion"] = np.where(
    weekly_panel["Registered_Units"] > 0,
    weekly_panel["Sold_Units"] / weekly_panel["Registered_Units"],
    np.nan
)
weekly_panel["Avg_Sale_Price"] = np.where(
    weekly_panel["Sale_Amount_Count"] > 0,
    weekly_panel["Sale_Amount_Sum"] / weekly_panel["Sale_Amount_Count"],
    np.nan
)
weekly_panel["RPU"] = np.where(
    weekly_panel["Sold_Units"] > 0,
    weekly_panel["BuyFee_Revenue"] / weekly_panel["Sold_Units"],
    np.nan
)
weekly_panel["Avg_BuyFee"] = np.where(
    weekly_panel["BuyFee_Count"] > 0,
    weekly_panel["BuyFee_Revenue"] / weekly_panel["BuyFee_Count"],
    np.nan
)

weekly_panel = weekly_panel.sort_values(["VRLOCATION", "LANE", "price_tier_low", "Week_Start"])
weekly_path = OUTPUT_DIR / "WEEKLY_MODEL_PANEL.csv"
weekly_panel.to_csv(weekly_path, index=False)

print("Saved:", weekly_path)
print("Weekly panel rows:", f"{len(weekly_panel):,}")
display(weekly_panel.head())


Saved: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_review_package\WEEKLY_MODEL_PANEL.csv
Weekly panel rows: 1,323,914


,Week_Start,VRLOCATION,LANE,price_tier,price_tier_low,price_tier_high,Registered_Units,Sold_Units,Sale_Amount_Sum,Sale_Amount_Count,BuyFee_Revenue,BuyFee_Count,Conversion,Avg_Sale_Price,RPU,Avg_BuyFee
570775,2023-01-02,ALBANY,A,0-1000,0.0,1000.0,4,4,2100.0,4,0.0,0,1.0,525.0,0.0,NaN
434628,2023-01-09,ALBANY,A,0-1000,0.0,1000.0,2,2,1000.0,2,0.0,0,1.0,500.0,0.0,NaN
711962,2023-01-16,ALBANY,A,0-1000,0.0,1000.0,3,0,0.0,0,0.0,0,0.0,NaN,NaN,NaN
1086030,2023-01-23,ALBANY,A,0-1000,0.0,1000.0,1,1,500.0,1,0.0,0,1.0,500.0,0.0,NaN
1080035,2023-01-30,ALBANY,A,0-1000,0.0,1000.0,1,1,500.0,1,0.0,0,1.0,500.0,0.0,NaN


## 7. Save `RANDOM_SAMPLE.csv`

In [12]:

if sample_parts:
    random_sample = pd.concat(sample_parts, ignore_index=True)
    if len(random_sample) > RANDOM_SAMPLE_SIZE:
        random_sample = random_sample.sample(n=RANDOM_SAMPLE_SIZE, random_state=RANDOM_STATE)
else:
    random_sample = pd.DataFrame()

random_path = OUTPUT_DIR / "RANDOM_SAMPLE.csv"
random_sample.to_csv(random_path, index=False)
print("Saved:", random_path)
print("Random sample rows:", f"{len(random_sample):,}")


Saved: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_review_package\RANDOM_SAMPLE.csv
Random sample rows: 100,000



## 8. Pass 2 — save `CHANGE_EVENT_SAMPLE.csv`

This re-reads the large CSV and keeps VIN rows within **±12 weeks** of genuine fee-change events at the same Location × Lane × Price Tier.


In [14]:

event_lookup = fee_events[["VRLOCATION", "LANE", "price_tier", "BUYFEE_CHANGE_DATE"]].dropna().copy()
event_lookup["BUYFEE_CHANGE_DATE"] = pd.to_datetime(event_lookup["BUYFEE_CHANGE_DATE"], errors="coerce")

event_date_map = {
    key: grp["BUYFEE_CHANGE_DATE"].sort_values().to_numpy(dtype="datetime64[ns]")
    for key, grp in event_lookup.groupby(["VRLOCATION", "LANE", "price_tier"], sort=False)
}

window_days = CHANGE_WINDOW_WEEKS * 7
change_sample_parts = []
rows_kept = 0
chunk_count_2 = 0

# reader2 = pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE, encoding=ENCODING, low_memory=False)
reader2 = pd.read_csv(
    INPUT_CSV,
    chunksize=CHUNK_SIZE,
    encoding=SAFE_ENCODING,
    encoding_errors="replace",
    low_memory=False
)

for chunk in reader2:
    chunk_count_2 += 1
    sale_dt = parse_date_series(chunk[COL["sale_date"]])
    sale_amt = to_num(chunk[COL["sale_amount"]])
    loc = clean_text(chunk[COL["location"]]).str.upper()
    lane = clean_text(chunk[COL["lane"]]).str.upper()
    _, _, tier_label = make_price_tier(sale_amt)

    keep = np.zeros(len(chunk), dtype=bool)
    temp_key = pd.DataFrame({"_loc": loc, "_lane": lane, "_tier": tier_label, "_date": sale_dt}, index=chunk.index)

    for key, idx_labels in temp_key.groupby(["_loc", "_lane", "_tier"], dropna=False).groups.items():
        dates = event_date_map.get(key)
        if dates is None or len(dates) == 0:
            continue

        idx_labels = np.asarray(list(idx_labels))
        obs_dates = temp_key.loc[idx_labels, "_date"].to_numpy(dtype="datetime64[ns]")
        valid = ~pd.isna(obs_dates)
        if not valid.any():
            continue

        obs_valid = obs_dates[valid]
        pos = np.searchsorted(dates, obs_valid)
        prev_pos = np.clip(pos - 1, 0, len(dates) - 1)
        next_pos = np.clip(pos, 0, len(dates) - 1)
        d1 = np.abs((obs_valid - dates[prev_pos]).astype("timedelta64[D]").astype(int))
        d2 = np.abs((obs_valid - dates[next_pos]).astype("timedelta64[D]").astype(int))
        mind = np.minimum(d1, d2)

        actual_labels = idx_labels[valid][mind <= window_days]
        positions = chunk.index.get_indexer(actual_labels)
        positions = positions[positions >= 0]
        keep[positions] = True

    if keep.any():
        kept = chunk.iloc[np.where(keep)[0]].copy()
        kept["_Derived_Sale_Date"] = sale_dt.iloc[np.where(keep)[0]].values
        kept["_Derived_Price_Tier"] = tier_label.iloc[np.where(keep)[0]].values
        change_sample_parts.append(kept)
        rows_kept += len(kept)

    if chunk_count_2 % 5 == 0:
        print(f"Pass 2: processed {chunk_count_2:,} chunks | kept {rows_kept:,} rows")

    del chunk, temp_key
    gc.collect()

if change_sample_parts:
    change_event_sample = pd.concat(change_sample_parts, ignore_index=True)
else:
    change_event_sample = pd.DataFrame()

change_sample_path = OUTPUT_DIR / "CHANGE_EVENT_SAMPLE.csv"
change_event_sample.to_csv(change_sample_path, index=False)

print("Saved:", change_sample_path)
print("Change-event sample rows:", f"{len(change_event_sample):,}")


Pass 2: processed 5 chunks | kept 152,612 rows
Pass 2: processed 10 chunks | kept 304,666 rows
Pass 2: processed 15 chunks | kept 455,894 rows
Pass 2: processed 20 chunks | kept 611,284 rows
Saved: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_review_package\CHANGE_EVENT_SAMPLE.csv
Change-event sample rows: 683,892


## 9. Save `COLUMN_SUMMARY.csv`

In [15]:

summary_rows = [
    {"Metric": "Total_Rows", "Value": total_rows},
    {"Metric": "Total_Columns", "Value": len(header_df.columns)},
    {"Metric": "Sale_Date_Min", "Value": min_sale_date},
    {"Metric": "Sale_Date_Max", "Value": max_sale_date},
    {"Metric": "Change_Date_Min", "Value": min_change_date},
    {"Metric": "Change_Date_Max", "Value": max_change_date},
    {"Metric": "Unique_Locations", "Value": len(global_location_values)},
    {"Metric": "Unique_Lanes", "Value": len(global_lane_values)},
    {"Metric": "Tracked_Unique_VINs", "Value": f">={VIN_UNIQUE_CAP:,}" if vin_unique_capped else len(global_vin_values)},
    {"Metric": "Rows_BUYFEE_CHANGE_STATUS_CHANGE", "Value": change_status_counter.get("CHANGE", 0)},
    {"Metric": "Rows_BUYFEE_CHANGE_STATUS_NO_CHANGE", "Value": change_status_counter.get("NO CHANGE", 0)},
    {"Metric": "Unique_Segment_Fee_Change_Events", "Value": len(fee_events)},
    {"Metric": "Unique_Location_Change_Dates", "Value": fee_events[["VRLOCATION", "BUYFEE_CHANGE_DATE"]].drop_duplicates().shape[0]},
    {"Metric": "Unique_Company_Change_Dates", "Value": fee_events["BUYFEE_CHANGE_DATE"].nunique()},
    {"Metric": "Weekly_Model_Panel_Rows", "Value": len(weekly_panel)},
    {"Metric": "Random_Sample_Rows", "Value": len(random_sample)},
    {"Metric": "Change_Event_Sample_Rows", "Value": len(change_event_sample)},
    {"Metric": "Price_Tier_Width", "Value": PRICE_TIER_WIDTH},
    {"Metric": "Change_Event_Window_Weeks", "Value": CHANGE_WINDOW_WEEKS},
]

for direction, count in fee_events["FEE_DIRECTION"].value_counts(dropna=False).items():
    summary_rows.append({"Metric": f"Fee_Events_{direction}", "Value": count})

column_summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "COLUMN_SUMMARY.csv"
column_summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)
display(column_summary)


Saved: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_review_package\COLUMN_SUMMARY.csv


,Metric,Value
0,Total_Rows,7540598
1,Total_Columns,36
2,Sale_Date_Min,2022-12-26 00:00:00
3,Sale_Date_Max,2026-12-25 00:00:00
4,Change_Date_Min,2023-01-08 00:00:00
5,Change_Date_Max,2026-02-08 00:00:00
6,Unique_Locations,54
7,Unique_Lanes,361
8,Tracked_Unique_VINs,">=2,000,000"
9,Rows_BUYFEE_CHANGE_STATUS_CHANGE,41252


## 10. Final QA / output manifest

In [16]:

outputs = {
    "DATA_PROFILE.csv": OUTPUT_DIR / "DATA_PROFILE.csv",
    "FEE_CHANGE_EVENTS.csv": OUTPUT_DIR / "FEE_CHANGE_EVENTS.csv",
    "WEEKLY_MODEL_PANEL.csv": OUTPUT_DIR / "WEEKLY_MODEL_PANEL.csv",
    "RANDOM_SAMPLE.csv": OUTPUT_DIR / "RANDOM_SAMPLE.csv",
    "CHANGE_EVENT_SAMPLE.csv": OUTPUT_DIR / "CHANGE_EVENT_SAMPLE.csv",
    "COLUMN_SUMMARY.csv": OUTPUT_DIR / "COLUMN_SUMMARY.csv",
}

print("=" * 90)
print("OUTPUT MANIFEST")
print("=" * 90)
for name, path in outputs.items():
    exists = path.exists()
    size_mb = path.stat().st_size / (1024**2) if exists else np.nan
    print(f"{name:<28} exists={str(exists):<5} size={size_mb:,.1f} MB")

print("\nReview package folder:")
print(OUTPUT_DIR)


OUTPUT MANIFEST
DATA_PROFILE.csv             exists=True  size=0.0 MB
FEE_CHANGE_EVENTS.csv        exists=True  size=0.7 MB
WEEKLY_MODEL_PANEL.csv       exists=True  size=93.5 MB
RANDOM_SAMPLE.csv            exists=True  size=17.9 MB
CHANGE_EVENT_SAMPLE.csv      exists=True  size=135.6 MB
COLUMN_SUMMARY.csv           exists=True  size=0.0 MB

Review package folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_review_package
